In [7]:
import json
import re
import yaml

patterns_map = dict(
    attemptId=r'attempt with (?:the )?id',
    categoryId=r'category with id',
    chapterId=r'chapter with id',
    choiceId=r'choice with id',
    commentId=r'comment with id',
    courseId=r'course with (?:the )?id|old course with id|new course with id|curso:',
    discussionId=r'discussion(?: with id)?',
    enrolmentId=r'enrolment method .*? with id',
    eventId=r'event .*? with id',
    evidenceId=r'evidence with id',
    fieldId=r'field with id',
    forumId=r'forum(?: with id)?',
    glossaryEntryId=r'glossary entry with id',
    gradeId=r'grade with id',
    gradeItemId=r'grade item with id',
    groupId=r'group with id',
    groupingId=r'grouping with id',
    h5pId=r'H5P with the id',
    itemId=r"Item(?: created)? with ID|item type ''.*?'' with id",
    moduleId=r'course module(?: with)? id',
    noteId=r'note with id',
    optionId=r'option with id',
    overrideId=r'override with id',
    pageId=r'page with (?:the )?id',
    postId=r'(?:forum )?post with id',
    questionId=r'question with id',
    questionCategoryId=r'question category with id',
    recordId=r'data record with id',
    roleId=r'role with id',
    ruleId=r'rule with id',
    scoId=r'sco with id',
    sectionId=r'section number|section with id',
    stepId=r'\(id',
    submissionId=r'submission(?: with id(?: of)?)?',
    subscriptionId=r'subscription(?: with id)?',
    tagId=r'tag with id',
    tourId=r'tour with id',
    userCompetencyId=r'user(?: course)? competency with id',
    userId=r'user with (?:the )?id|user',
    fileCount=r'uploaded',
    stepIndex=r'step index',
    wordCount=r'submission with',
    ratingValue=r"with(?= ''INTEGER'' rating)",
    scormValue=r'value of'
)

# Ordenamos de mayor a menor longitud para evitar colisiones de prefijos
sorted_patterns = sorted(patterns_map.items(), key=lambda x: len(x[1]), reverse=True)

def transform_template(text):
    def replacer(match):
        start_pos = match.start()
        # Eliminamos comillas y espacios sobrantes para que el anclaje $ funcione correctamente
        subtext = text[:start_pos].rstrip("'\" ")
        
        for field_name, pattern in sorted_patterns:
            if re.search(pattern + r'$', subtext, re.IGNORECASE):
                return f"(?<{field_name}>-?\\d+)"
        return "(-?\\d+)" # Fallback seguro si no encuentra coincidencia exacta

    # Sustituye únicamente la palabra INTEGER por el grupo con nombre
    return re.sub(r"INTEGER", replacer, text)

def transform(node):
    if isinstance(node, dict):
        return {k: transform(v) for k, v in node.items()}
    elif isinstance(node, list):
        return [transform_template(item) for item in node]
    return node

with open('../components.json') as f:
    data = json.load(f)

mappings = transform(data)

print(yaml.dump(mappings, default_flow_style=False, sort_keys=True, allow_unicode=True, width=float("inf")))

Activity report:
  Activity report viewed:
  - The user with id '(?<userId>-?\d+)' viewed the outline activity report for the course with id '(?<courseId>-?\d+)'.
  Outline report viewed:
  - The user with id '(?<userId>-?\d+)' viewed the outline report for the user with id '(?<userId>-?\d+)' for the course with id '(?<courseId>-?\d+)'.
Assignment:
  A submission has been submitted.:
  - The user with id '(?<userId>-?\d+)' has submitted the submission with id '(?<submissionId>-?\d+)' for the assignment with course module id '(?<moduleId>-?\d+)'.
  All the submissions are being downloaded.:
  - The user with id '(?<userId>-?\d+)' has downloaded all the submissions for the assignment with course module id '(?<moduleId>-?\d+)'.
  An extension has been granted.:
  - The user with id '(?<userId>-?\d+)' has granted an extension for the user with id '(?<userId>-?\d+)' for the assignment with course module id '(?<moduleId>-?\d+)'.
  Assignment override created:
  - The user with id '(?<userId>